# 🏛️ Dual-GPU Council: State-Aware Test Generation Benchmark
**Hardware Target:** Kaggle Dual-GPU (2x Tesla T4 15GB) — bash Compute Cost
**Scope:** Evaluates Coder (GPU 0) + Critic (GPU 1) across pure-Python targets from Test4Py Table 1 (NA Projects).

---
### Quick Start:
1. In the Kaggle sidebar, set **Accelerator** -> **GPU T4 x2**.
2. Click **Run All** (Total execution time: ~35-40 minutes).
3. Output artifacts and  will be generated in .

In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP & DEPENDENCIES
# ==============================================================================
import os, sys, subprocess

print("⏳ [1/2] Installing required libraries...")
pkgs = ["transformers>=4.40.0", "accelerate>=0.29.0", "bitsandbytes>=0.43.0", 
        "pytest", "pytest-json-report", "coverage", "numpy"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)

print("⏳ [2/2] Verifying GPU setup...")
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  - GPU {i}: {torch.cuda.get_device_name(i)}")

assert torch.cuda.device_count() >= 2, "⚠️ Please enable GPU T4 x2 accelerator in Kaggle settings!"
print("
✅ Environment ready for Dual-GPU Council execution!")

In [ ]:
# ==============================================================================
# CELL 2: CLONE OR UPDATE RESEARCH REPOSITORY
# ==============================================================================
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/nguyendevwk/Test4Py-Setup-pilot-temp.git"
WORKDIR = Path("/kaggle/working/Test4Py_Setup")

if not WORKDIR.exists():
    print(f"📥 Cloning research repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
else:
    print("\ud83d\udd04 Updating repository to latest commit...")
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)

os.chdir(WORKDIR)
print(f"
📁 Working directory set to: {os.getcwd()}")

In [ ]:
# ==============================================================================
# CELL 3: EXECUTE MULTI-TARGET STANDARDIZED BENCHMARK
# ==============================================================================
import sys, subprocess

# Running 3-Arm comparison across curated pure-Python targets
# Arms: baseline (No-Setup Control) vs ranked (Static Heuristic) vs verified (Dynamic State Probe)
cmd = [
    sys.executable, "council_multitarget.py",
    "--output", "/kaggle/working/EVALUATION_OUTPUT",
    "--repos-dir", "/kaggle/working/BENCHMARK_REPOS",
    "--targets", "all",
    "--arms", "baseline,ranked,verified",
    "--samples", "2",
    "--temperature", "0.7",
    "--coder-model", "Qwen/Qwen2.5-Coder-7B-Instruct",
    "--critic-model", "Qwen/Qwen3-4B-Instruct-2507"
]

print("🚀 Starting Benchmark execution... This will take ~35-40 minutes.")
proc = subprocess.run(cmd, check=True)
print("\n\ud83c\udf89 Benchmark finished successfully!")

In [ ]:
# ==============================================================================
# CELL 4: DISPLAY REPORT & DOWNLOAD ARTIFACTS
# ==============================================================================
from IPython.display import display, Markdown, FileLink
from pathlib import Path

report_file = Path("/kaggle/working/EVALUATION_OUTPUT/FINAL_REPORT.md")
if report_file.exists():
    display(Markdown(report_file.read_text()))

zip_file = Path("/kaggle/working/EVALUATION_OUTPUT.zip")
if zip_file.exists():
    print(f"\n\ud83d\udce5 Download your results here: {zip_file}")
    display(FileLink(str(zip_file)))